In [1]:
import os
import json
import shutil
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
NUM_TOPICS = 20  # TODO: this value does not affect anything

NUM_ITERATIONS = 5
NUM_TOP_TOKENS = 20
NUM_TRAINS = 20

In [6]:
BERTOPIC_FOLDER_PATH = '/data_mil/shared/CompressaAI/BERTopic'

In [7]:
! ls $BERTOPIC_FOLDER_PATH

results  results50


In [8]:
! ls $BERTOPIC_FOLDER_PATH/results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [9]:
RESULTS_FOLDER_PATH = os.path.join(BERTOPIC_FOLDER_PATH, 'results', 'mkb10')

In [10]:
RESULTS_FOLDER_PATH

'/data_mil/shared/CompressaAI/BERTopic/results/mkb10'

In [11]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [12]:
SAVE_FOLDER = os.path.join('results', 'mkb10')

In [13]:
SAVE_FOLDER

'results/mkb10'

In [14]:
! ls $SAVE_FOLDER

ablation_study		iterative_100000.json	    lda.json
decorrelation.json	iterative2_100000000	    plsa.json
iterative_100000	iterative2_1000000000	    sparse.json
iterative_1000000	iterative2_1000000000.json  tless.json
iterative_1000000.json	iterative2_100000000.json


In [15]:
! ls $RESULTS_FOLDER_PATH

0  1  10  11  12  13  14  15  16  17  18  19  2  3  4  5  6  7	8  9


In [18]:
! ls $RESULTS_FOLDER_PATH/0

dataset.csv  dataset__internals  phi.csv  top_words.json


In [17]:
dataset = Dataset(
    f'{RESULTS_FOLDER_PATH}/0/dataset.csv',
)

dataset.get_possible_modalities()

{'@text'}

In [19]:
MAIN_MODALITY = '@text'

In [22]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [23]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 10.5 s, sys: 533 ms, total: 11.1 s
Wall time: 10.9 s


In [24]:
co_occurences.shape

(127266, 127266)

In [25]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [26]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [27]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [28]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, topic_names: List[str], parent_model=None, parent_phi=None, words=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._words = words
        self._word_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = self._parent_phi
            vals = parent_phi.values
        else:
            assert False

            parent_phi = self._parent_model.get_phi()
            vals = parent_phi.values[:, self._topic_indices]

        if self._word_indices is None:
            assert vals.shape[0] == rwt.shape[0], (vals.shape[0], rwt.shape[0])
        else:
            assert vals.shape[0] == len(self._word_indices)

        assert vals.shape[1] == len(self._topic_indices)

        if self._word_indices is None:
            rwt[:, self._topic_indices] += vals
        else:
            # print(len(self._word_indices), len(self._topic_indices), rwt.shape, vals.shape)
            # print(self._word_indices[:3], self._topic_indices[:3])
            # print(rwt[np.ix_(self._word_indices, self._topic_indices)].shape, vals.shape)

            # https://github.com/numpy/numpy/issues/5574
            # https://github.com/numpy/numpy/issues/13255
            rwt[np.ix_(self._word_indices, self._topic_indices)] += vals

            # print(np.ix_(self._word_indices, self._topic_indices)[:3])

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

        if self._words is not None:
            self._word_indices = [
                # phi.index.get_loc(w) for w in self._words
                int(phi.index.get_loc(w)) for w in self._words
            ]

            # print(f'!!! Word indices: {self._word_indices}')

In [29]:
def fit_and_compute_scores(model, dataset, target_topic_indices=None, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account

    if target_topic_indices is None:
        target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()

    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [30]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [31]:
TOPIC_INDICES = list(range(NUM_TOPICS))

In [32]:
TOPIC_INDICES

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [33]:
phi0 = pd.read_csv(f'{RESULTS_FOLDER_PATH}/0/phi.csv', index_col=0)

In [34]:
phi0.head()

,background_1,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,...,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15,topic_16,topic_17,topic_18,topic_19
00,0.000000,0.000060,0.000000,0.000856,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001954,0.0,0.000000,0.0
000,0.002126,0.001494,0.001659,0.000527,0.000908,0.002484,0.001082,0.002825,0.000223,0.000763,...,0.001287,0.001975,0.002378,0.000557,0.020781,0.001586,0.000000,0.0,0.006863,0.0
0000001,0.000000,0.000000,0.000035,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
00002,0.000000,0.000000,0.000035,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
0001,0.000000,0.000000,0.000061,0.000000,0.000171,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0


In [35]:
phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

In [36]:
phi0.head()

background_1   topic_0   topic_1   topic_2   topic_3   topic_4  \
@text 00           0.000000  0.000060  0.000000  0.000856  0.000000  0.000000   
      000          0.002126  0.001494  0.001659  0.000527  0.000908  0.002484   
      0000001      0.000000  0.000000  0.000035  0.000000  0.000000  0.000000   
      00002        0.000000  0.000000  0.000035  0.000000  0.000000  0.000000   
      0001         0.000000  0.000000  0.000061  0.000000  0.000171  0.000000   

                topic_5   topic_6   topic_7   topic_8  ...  topic_10  \
@text 00       0.000000  0.000000  0.000000  0.000000  ...  0.000000   
      000      0.001082  0.002825  0.000223  0.000763  ...  0.001287   
      0000001  0.000000  0.000000  0.000000  0.000000  ...  0.000000   
      00002    0.000000  0.000000  0.000000  0.000000  ...  0.000000   
      0001     0.000000  0.000000  0.000000  0.000000  ...  0.000000   

               topic_11  topic_12  topic_13  topic_14  topic_15  topic_16  \
@text 00       0.000000  0.000000  0.000000  0.000000  0.000000  0.001954   
      000      0.001975  0.002378  0.000557  0.020781  0.001586  0.000000   
      0000001  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
      00002    0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
      0001     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   

               topic_17  topic_18  topic_19  
@text 00            0.0  0.000000       0.0  
      000           0.0  0.006863       0.0  
      0000001       0.0  0.000000       0.0  
      00002         0.0  0.000000       0.0  
      0001          0.0  0.000000       0.0  

[5 rows x 21 columns]

In [40]:
! head -n 10 $RESULTS_FOLDER_PATH/0/top_words.json

{
    "background_1": [
        [
            "мозга",
            0.012196034987323405
        ],
        [
            "шока",
            0.011068954790480465
        ],


In [41]:
DIFF_THRESHOLD = 2

In [42]:
def check_top_words(phi, top_words):
    diffs = []

    for t, topic_top_words in top_words.items():
        if t == 'background_1':
            print(f'Skipping background topic: {t}.')

            continue

        print(t)
        # print(top_words)
    
        top_phi = set(phi[t].sort_values(ascending=False)[:NUM_TOP_TOKENS].index.get_level_values(1))
        top_bt = set([p[0] for p in topic_top_words])
    
        if top_phi == top_bt:
            diffs.append(
                {
                    'total': 0,
                    'lost_bt': 0,
                    'lost_model': 0,
                }
            )
        else:
            diff1 = top_phi.difference(top_bt)
            diff2 = top_bt.difference(top_phi)
    
            print('  WTF:', diff1, diff2)
    
            if len(diff1) > DIFF_THRESHOLD:
                print(f'  WTF?!?!?', len(diff1))
    
            if len(diff2) > DIFF_THRESHOLD:
                print(f'  WTF?!?!?', len(diff2))

            diffs.append(
                {
                    'total': len(diff1 | diff2),
                    'lost_bt': len(diff2),
                    'lost_model': len(diff1),
                }
            )

    return diffs

In [43]:
def init_model_and_phi(num_topics, dataset, phi0):
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=num_topics,
        seed=0,
    )
    
    phi = model.get_phi()
    
    # assert phi.shape[1] == phi0.shape[1] - 1
    assert phi.shape[1] == num_topics
    assert phi.shape[1] in [phi0.shape[1], phi0.shape[1] - 1]
    
    common_words = list(set(phi.index).intersection(phi0.index))
    
    phi.loc[:, :] = 0

    if phi.shape[1] == phi0.shape[1]:
        target_topics = phi0.columns
    elif phi.shape[1] == phi0.shape[1] - 1:
        target_topics = phi.columns
    else:
        assert False

    # print(common_words, phi.index, phi0.index)

    phi.loc[common_words, :] += phi0.loc[common_words, target_topics]

    diff = set(phi0.columns).symmetric_difference(set(phi.columns))

    assert diff == {'background_1'} or diff == {'background_1', f'topic_{len(phi.columns) - 1}'}
    
    phi = phi / phi.sum(axis=0)

    return model, phi, common_words

In [44]:
results = []

is_results_loaded = False

save_file_path = os.path.join(
    SAVE_FOLDER, 'bertopic.json'
)

if os.path.isfile(save_file_path):
    results = json.loads(
        open(save_file_path).read()
    )
    is_results_loaded = True


singles_save_folder = os.path.join(
    SAVE_FOLDER, 'bertopic'
)

os.makedirs(singles_save_folder, exist_ok=True)


for seed in range(NUM_TRAINS):
    if is_results_loaded:
        continue

    print(seed)

    current_save_file_path = os.path.join(
        singles_save_folder, f'bertopic_{seed}.json'
    )

    if os.path.isfile(current_save_file_path):
        print(f'Already computed results: {current_save_file_path}. Loading and skipping...')

        result = json.loads(
            open(current_save_file_path).read()
        )
        results.append(result)

        continue

    seed_load_folder = os.path.join(RESULTS_FOLDER_PATH, str(seed))
    files = [f for f in os.listdir(seed_load_folder) if os.path.isfile(f'{seed_load_folder}/{f}')]
    
    assert len(files) == 3

    dataset = Dataset(
        f'{seed_load_folder}/dataset.csv',
    )
    phi0 = pd.read_csv(f'{seed_load_folder}/phi.csv', index_col=0)
    phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

    assert all('back' not in t for t in phi0.columns[1:])
    assert 'back' in phi0.columns[0]

    num_specific_topics = phi0.shape[1] - 1

    assert num_specific_topics == len([t for t in phi0.columns if 'back' not in t])

    print(f'Num model topics: {phi0.shape[1]}.')

    with open(f'{seed_load_folder}/top_words.json', 'r') as f:
        top_words = json.loads(f.read())


    
    model, phi, common_phi_words = init_model_and_phi(
        num_topics=num_specific_topics + 1,  # background
        dataset=dataset, phi0=phi0
    )
    common_words = [
        w[1] for w in common_phi_words  # without modality
    ]
    common_words_phi0_indices = [
        # phi0.index.get_loc(w) for w in common_words
        int(phi0.index.get_locs(w)) for w in common_phi_words
    ]

    print('Check before fit:')
    diff_tops1 = check_top_words(phi, top_words)  # Whatever...
    
    fix_regularizer = FastFixPhiRegularizer(
        name='fix',
        parent_phi=phi0.iloc[common_words_phi0_indices, 1:],
        words=common_words,
        topic_names=phi.columns[1:],
    )
    result = fit_and_compute_scores(
        model, dataset,
        target_topic_indices=list(range(phi.shape[1])),
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )

    print('Check after fit:')
    diff_tops2 = check_top_words(phi, top_words)  # Whatever...

    assert diff_tops1 == diff_tops2

    fair_ppl_free = result['scores']['perplexity']

    del model, phi, fix_regularizer, result


    
    model, phi, _ = init_model_and_phi(
        num_topics=num_specific_topics + 1,  # background
        dataset=dataset, phi0=phi0
    )
    fix_regularizer = FastFixPhiRegularizer(
        name='fix',
        parent_phi=phi0.iloc[common_words_phi0_indices, :],  # Diff here
        words=common_words,
        topic_names=phi.columns,
    )
    result = fit_and_compute_scores(
        model, dataset,
        target_topic_indices=list(range(phi.shape[1])),
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )
    
    fair_ppl_fix = result['scores']['perplexity']

    del model, phi, fix_regularizer, result

    
    
    model, phi, _ = init_model_and_phi(
        num_topics=num_specific_topics,  # Diff here
        dataset=dataset, phi0=phi0
    )
    fix_regularizer = FastFixPhiRegularizer(
        name='fix',
        parent_phi=phi0.iloc[common_words_phi0_indices, 1:],  # Diff here
        words=common_words,
        topic_names=phi.columns,
    )
    result = fit_and_compute_scores(
        model, dataset,
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )

    unfair_ppl_banklike = result['scores']['perplexity']


    
    result['scores']['fair_ppl_free'] = fair_ppl_free
    result['scores']['fair_ppl_fix'] = fair_ppl_fix
    result['scores']['unfair_ppl_banklike'] = unfair_ppl_banklike

    assert result['scores']['fair_ppl_free'] < result['scores']['unfair_ppl_banklike']
    # assert result['scores']['fair_ppl_fix'] < result['scores']['unfair_ppl_banklike']

    result['scores']['coherence_20'] = float(result['scores']['coherence_20'])
    result['stats'] = {
        'num_topics': phi0.shape[1],
        'num_common_words': len(common_words),
        'num_model_words': phi.shape[0],
        'num_bt_words': phi0.shape[0],
        'top_diffs': diff_tops2,
    }

    results.append(result)

    print(result['scores'])
    print(result['stats'])

    dumped_result = json.dumps(
        results[-1], indent=4
    )

    with open(current_save_file_path, 'w') as f:
        f.write(dumped_result)

    del model, phi, fix_regularizer


with open(save_file_path, 'w') as f:
    f.write(
        json.dumps(
            results, indent=4
        )
    )

0
Num model topics: 21.


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



Check before fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
  WTF: {'симптомы', 'сна', 'детей'} {'мкб', 'dsm', 'аспергера'}
  WTF?!?!? 3
  WTF?!?!? 3
topic_3
topic_4
  WTF: {'раком'} {'гисо'}
topic_5
topic_6
  WTF: {'также', 'лечение'} {'экг', 'ибс'}
topic_7
topic_8
  WTF: {'анемией', 'анемия', 'лечения'} {'гит', 'виллебранда', 'viii'}
  WTF?!?!? 3
  WTF?!?!? 3
topic_9
  WTF: {'миелоидного', 'это', 'лимфома'} {'опл', 'd1', 'омл'}
  WTF?!?!? 3
  WTF?!?!? 3
topic_10
topic_11
  WTF: {'энурез', 'ожирения', 'является'} {'мкб', 'commentedtext', 'dsm'}
  WTF?!?!? 3
  WTF?!?!? 3
topic_12
topic_13
topic_14
  WTF: {'также', 'обычно', 'две', 'мужчин', 'мужчины', 'анеуплоидия', 'хромосому'} {'xxyy', 'фабри', 'магенис', 'xx', 'смит', 'xxxy', 'xy'}
  WTF?!?!? 7
  WTF?!?!? 7
topic_15
  WTF: {'содержание'} {'адг'}
topic_16
topic_17
  WTF: {'приона', 'парапроктита', 'прионные', 'году'} {'prp', 'prpsc', 'prpc', 'paragonimus'}
  WTF?!?!? 4
  WTF?!?!? 4
topic_18
  WTF: {'still', 'бо

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fba55142b20>}


...

Check before fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
  WTF: {'симптомы', 'детей', 'это', 'развития'} {'мкб', 'dsm', 'аспергера', 'начало_цитаты'}
  WTF?!?!? 4
  WTF?!?!? 4
topic_3
  WTF: {'раком'} {'гисо'}
topic_4
topic_5
  WTF: {'лечение', 'инфаркта'} {'экг', 'ибс'}
topic_6
  WTF: {'лечение'} {'срк'}
topic_7
topic_8
  WTF: {'миелоидного', 'это', 'лимфома'} {'опл', 'd1', 'омл'}
  WTF?!?!? 3
  WTF?!?!? 3
topic_9
topic_10
  WTF: {'энурез', 'ожирения', 'является'} {'мкб', 'commentedtext', 'dsm'}
  WTF?!?!? 3
  WTF?!?!? 3
topic_11
topic_12
topic_13
topic_14
  WTF: {'также', 'обычно', 'две', 'мужчин', 'мужчины', 'анеуплоидия', 'хромосому'} {'xxyy', 'фабри', 'магенис', 'xx', 'смит', 'xxxy', 'xy'}
  WTF?!?!? 7
  WTF?!?!? 7
topic_15
  WTF: {'содержание'} {'адг'}
topic_16
topic_17
  WTF: {'приона', 'парапроктита', 'прионные', 'году'} {'prp', 'prpsc', 'prpc', 'paragonimus'}
  WTF?!?!? 4
  WTF?!?!? 4
topic_18
  WTF: {'still', 'больных', '50', 'являются'} {'b27', 'цог

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fba411af0d0>}


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fba30426520>}
{'perplexity': 52801.13671875, 'coherence_20': 1.3051093832264393, 'diversity_euclidean': 0.030266341228989045, 'diversity_jensenshannon': 0.685966892878748, 'diversity_hellinger': 0.8099164707509697, 'diversity_cosine': 0.7291960027138465, 'fair_ppl_free': 8277.4697265625, 'fair_ppl_fix': 51219.421875, 'unfair_ppl_banklike': 52801.13671875}
{'num_topics': 21, 'num_common_words': 101018, 'num_model_words': 127266, 'num_bt_words': 115899, 'top_diffs': [{'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 8, 'lost_bt': 4, 'lost_model': 4}, {'total': 2, 'lost_bt': 1, 'lost_model': 1}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 4, 'lost_bt': 2, 'lost_model': 2}, {'total': 2, 'lost_bt': 1, 'lost_model': 1}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 6, 'lost_bt': 3, 'lost_model': 3}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 6, 'lost_bt': 3, 

In [46]:
1

1